# 4 — Vertex AI Search (Discovery Engine)

**AWS-side equivalent:** Bedrock Knowledge Base on OpenSearch / Aurora
**GCP service:** Vertex AI Search (rebranded "Agent Search" in 2026 — same
API surface, same SDK module name `google.cloud.discoveryengine`).

AWS-side notebook: `notebooks/aws/04-knowledge-base.ipynb`

> **Naming note (verified May 2026):** Google renamed "Vertex AI Search"
> to **Agent Search**, but resource URIs and Python SDK module names are
> unchanged (`google-cloud-discoveryengine`, `dataStores/...`, `engines/...`).
> We use the API names verbatim throughout.

## Steps

1. Create a GCS bucket and upload a handful of sample org docs (same content
   shape as AWS notebook 04: kubernetes-migration, ml-pipeline, zero-trust-auth).
2. Create a Discovery Engine **data store** (unstructured, GENERIC vertical).
3. Import the GCS docs into the data store (LRO — wait for completion).
4. Create a Discovery Engine **search engine** (the "app" / serving config).
5. Hit `SearchService.search` from Python.
6. Cross-reference with AWS KB notebook (`retrieve` shape + score field).

## Step 1: Load config + decide resource names

In [ ]:
import json
from pathlib import Path
from cleanup_utils import ResourceTracker

config_path = Path('config.json')
config = json.loads(config_path.read_text())
gcp = config.setdefault('gcp', {})
PROJECT_ID = gcp.get('project_id')
REGION = gcp.get('region', 'us-central1')
assert PROJECT_ID, 'Run 01-setup-credentials.ipynb first.'

# Discovery Engine resources live in `global` (and a few US/EU multi-regions);
# Vertex AI lives in regions. They are independent — don't conflate.
DE_LOCATION = gcp.setdefault('discovery_engine_location', 'global')

DOCS_BUCKET = gcp.setdefault('docs_bucket', f'{PROJECT_ID}-orgiq-documents')
DATA_STORE_ID = gcp.setdefault('discovery_data_store_id', 'orgiq-docs')
ENGINE_ID = gcp.setdefault('discovery_engine_id', 'orgiq-search')

print(f'project           : {PROJECT_ID}')
print(f'GCS bucket        : {DOCS_BUCKET}')
print(f'DE location       : {DE_LOCATION}')
print(f'data store id     : {DATA_STORE_ID}')
print(f'engine id         : {ENGINE_ID}')

tracker = ResourceTracker()

## Step 2: Create the GCS bucket + upload sample docs

Same three sample docs used by the AWS notebook so we can compare retrieval
quality apples-to-apples. We store them as `.md` — Discovery Engine handles
Markdown natively.

In [ ]:
from google.cloud import storage
from google.api_core.exceptions import Conflict

gcs = storage.Client(project=PROJECT_ID)

try:
    # Bucket location is independent from DE location. US multi-region is
    # the standard choice for ingestion → DE global.
    bucket = gcs.create_bucket(DOCS_BUCKET, location='US')
    print(f'✓ Created bucket gs://{DOCS_BUCKET}')
except Conflict:
    bucket = gcs.bucket(DOCS_BUCKET)
    print(f'  Bucket gs://{DOCS_BUCKET} already exists')

tracker.track('gcs_bucket', DOCS_BUCKET)
tracker.flush()

SAMPLE_DOCS = [
    {
        'key': 'docs/kubernetes-migration.md',
        'content': """# Kubernetes Migration Playbook

This guide covers migrating monolithic applications to Kubernetes (GKE).

## Prerequisites
- Container runtime (Docker / containerd)
- kubectl configured against the target GKE cluster
- 3+ node pool

## Steps
1. Containerize the application with multi-stage Dockerfiles
2. Define resource requests and limits
3. Create Deployment and Service manifests
4. Configure Horizontal Pod Autoscaler
5. Set up Ingress with managed certs (cert-manager)

## Best Practices
- Use namespaces for isolation
- Implement readiness and liveness probes
- Store secrets in Secret Manager (mounted via CSI driver)
- Use the cluster autoscaler + node auto-provisioning
""",
    },
    {
        'key': 'docs/ml-pipeline-architecture.md',
        'content': """# ML Pipeline Architecture

End-to-end machine learning pipeline using Vertex AI Pipelines.

## Components
- Data ingestion: GCS + Dataflow
- Feature store: Vertex AI Feature Store
- Training: Vertex AI Training jobs
- Evaluation: Vertex AI Experiments + custom metrics
- Deployment: Vertex AI Endpoints with traffic splitting

## Model Registry
All models versioned in Vertex AI Model Registry with approval workflows.

## Monitoring
- Model drift with Vertex AI Model Monitoring
- A/B testing via endpoint traffic splits
- Cloud Monitoring custom metrics for business KPIs
""",
    },
    {
        'key': 'docs/zero-trust-auth.md',
        'content': """# Zero Trust Authentication Design

Implementing zero trust security for microservices on GKE.

## Principles
- Never trust, always verify
- Least privilege access
- Assume breach

## Implementation
1. mTLS between all services (Istio / Anthos Service Mesh)
2. JWT validation at every service boundary
3. Short-lived tokens (15 min expiry)
4. Device trust scoring (BeyondCorp Enterprise)
5. Continuous authorization checks

## Token Architecture
- Identity Provider: Identity Platform / Firebase Auth
- Access tokens: RS256 signed JWTs
- Refresh tokens: stored in Firestore with rotation
- Service tokens: Workload Identity for service accounts
""",
    },
]

for doc in SAMPLE_DOCS:
    blob = bucket.blob(doc['key'])
    blob.upload_from_string(doc['content'], content_type='text/markdown')
    print(f"  ✓ uploaded gs://{DOCS_BUCKET}/{doc['key']}")

## Step 3: Create the Discovery Engine data store

`industry_vertical=GENERIC` is the right choice for org docs (vs MEDIA,
RETAIL, etc.). `content_config=CONTENT_REQUIRED` says "unstructured docs"
(vs `NO_CONTENT` for metadata-only).

In [ ]:
from google.api_core.client_options import ClientOptions
from google.api_core.exceptions import AlreadyExists
from google.cloud import discoveryengine_v1 as discoveryengine

# Discovery Engine endpoints are regional. `global` uses the default endpoint;
# `us` / `eu` need the matching `<loc>-discoveryengine.googleapis.com`.
client_options = (
    ClientOptions(api_endpoint=f'{DE_LOCATION}-discoveryengine.googleapis.com')
    if DE_LOCATION != 'global' else None
)

ds_client = discoveryengine.DataStoreServiceClient(client_options=client_options)
ds_parent = ds_client.collection_path(
    project=PROJECT_ID, location=DE_LOCATION, collection='default_collection'
)
ds_resource = f'{ds_parent}/dataStores/{DATA_STORE_ID}'

data_store_req = discoveryengine.CreateDataStoreRequest(
    parent=ds_parent,
    data_store_id=DATA_STORE_ID,
    data_store=discoveryengine.DataStore(
        display_name='OrgIQ docs',
        industry_vertical=discoveryengine.IndustryVertical.GENERIC,
        solution_types=[discoveryengine.SolutionType.SOLUTION_TYPE_SEARCH],
        content_config=discoveryengine.DataStore.ContentConfig.CONTENT_REQUIRED,
    ),
)

try:
    op = ds_client.create_data_store(request=data_store_req)
    ds = op.result(timeout=300)
    print(f'✓ Created data store: {ds.name}')
except AlreadyExists:
    print(f'  Data store already exists: {ds_resource}')

tracker.track('discovery_engine_data_store', DATA_STORE_ID, location=DE_LOCATION)
tracker.flush()

## Step 4: Import the GCS docs (LRO)

`reconciliation_mode=FULL` mirrors GCS into the data store on every import
(removes anything not in the new manifest). Use `INCREMENTAL` if you want
append-only behaviour. We use `INCREMENTAL` here so re-runs are cheap.

In [ ]:
doc_client = discoveryengine.DocumentServiceClient(client_options=client_options)
branch = doc_client.branch_path(
    project=PROJECT_ID,
    location=DE_LOCATION,
    data_store=DATA_STORE_ID,
    branch='default_branch',
)

import_req = discoveryengine.ImportDocumentsRequest(
    parent=branch,
    gcs_source=discoveryengine.GcsSource(
        input_uris=[f'gs://{DOCS_BUCKET}/docs/*'],
        # `content` = unstructured files (pdf/html/md/txt etc).
        data_schema='content',
    ),
    reconciliation_mode=discoveryengine.ImportDocumentsRequest.ReconciliationMode.INCREMENTAL,
)

print('⏳ Starting import LRO (this can take 1–5 min)…')
import_op = doc_client.import_documents(request=import_req)
import_result = import_op.result(timeout=900)
print('✓ Import complete')
# `metadata` is an ImportDocumentsMetadata proto with success_count + failure_count.
md = import_op.metadata
if md is not None:
    print(f'  success_count={getattr(md, "success_count", "?")} failure_count={getattr(md, "failure_count", "?")}')

## Step 5: Create the search engine (app)

An "engine" in Discovery Engine ≈ Bedrock KB's search app. It binds one or
more data stores to a serving config (the thing you actually `search`).

In [ ]:
engine_client = discoveryengine.EngineServiceClient(client_options=client_options)
engine_req = discoveryengine.CreateEngineRequest(
    parent=ds_parent,
    engine_id=ENGINE_ID,
    engine=discoveryengine.Engine(
        display_name='OrgIQ search',
        solution_type=discoveryengine.SolutionType.SOLUTION_TYPE_SEARCH,
        industry_vertical=discoveryengine.IndustryVertical.GENERIC,
        data_store_ids=[DATA_STORE_ID],
        search_engine_config=discoveryengine.Engine.SearchEngineConfig(
            # STANDARD tier covers plain retrieval at the lowest price. Upgrade
            # to SEARCH_TIER_ENTERPRISE + add `search_add_ons=[SEARCH_ADD_ON_LLM]`
            # if you want generative summaries / follow-up suggestions.
            search_tier=discoveryengine.SearchTier.SEARCH_TIER_STANDARD,
        ),
    ),
)

try:
    op = engine_client.create_engine(request=engine_req)
    eng = op.result(timeout=600)
    print(f'✓ Created engine: {eng.name}')
except AlreadyExists:
    print(f'  Engine already exists: {ENGINE_ID}')

tracker.track('discovery_engine_engine', ENGINE_ID, location=DE_LOCATION)
tracker.flush()

## Step 6: Run a search query

`serving_config` lives under the engine. `default_search` is the auto-created
config — most apps never need a second one.

In [ ]:
search_client = discoveryengine.SearchServiceClient(client_options=client_options)
serving_config = search_client.serving_config_path(
    project=PROJECT_ID,
    location=DE_LOCATION,
    data_store=DATA_STORE_ID,
    serving_config='default_search',
)

QUERY = 'How do I migrate applications to Kubernetes?'
req = discoveryengine.SearchRequest(
    serving_config=serving_config,
    query=QUERY,
    page_size=5,
    content_search_spec=discoveryengine.SearchRequest.ContentSearchSpec(
        snippet_spec=discoveryengine.SearchRequest.ContentSearchSpec.SnippetSpec(
            return_snippet=True,
        ),
        # Generative summaries require ENTERPRISE tier + SEARCH_ADD_ON_LLM. Add
        # back a `summary_spec=...` here once the engine is upgraded; for plain
        # snippet retrieval STANDARD tier is sufficient.
    ),
)

resp = search_client.search(req)
print(f'Query: {QUERY!r}\n')
for i, r in enumerate(resp.results, 1):
    doc = r.document
    derived = dict(doc.derived_struct_data) if doc.derived_struct_data else {}
    snippets = derived.get('snippets', [])
    snippet = snippets[0].get('snippet') if snippets else '(no snippet)'
    print(f'  {i}. {doc.id}')
    print(f'     {snippet[:140]}…' if snippet else '')

if getattr(resp, 'summary', None) and getattr(resp.summary, 'summary_text', ''):
    print(f'\n--- LLM summary ---\n{resp.summary.summary_text}')
else:
    print('\n(no summary — STANDARD tier; upgrade to ENTERPRISE + SEARCH_ADD_ON_LLM for generative summaries)')

## Step 7: Cross-reference with AWS-side `retrieve()` shape

Bedrock KB returns `retrievalResults[].content.text` and `score`. Discovery
Engine returns `results[].document.derived_struct_data['snippets']` and a
relevance score on `r.document.derived_struct_data['relevance_score']`
(populated when `relevance_score_spec` is on the request). The cell below
normalizes both to a common dict so downstream agent code can swap clouds.

In [ ]:
def normalize_de_results(resp) -> list[dict]:
    out = []
    for r in resp.results:
        derived = dict(r.document.derived_struct_data) if r.document.derived_struct_data else {}
        snippets = derived.get('snippets') or []
        snippet_text = snippets[0].get('snippet') if snippets else ''
        score = derived.get('relevance_score')
        out.append({
            'id': r.document.id,
            'uri': derived.get('link', ''),
            'text': snippet_text,
            'score': float(score) if score is not None else None,
        })
    return out

for hit in normalize_de_results(resp):
    print(f"  [{hit['id']}]  score={hit['score']}  uri={hit['uri']}")
    print(f"    text: {hit['text'][:140]}…")

## Step 8: Persist resource IDs to `config.json`

In [ ]:
gcp['docs_bucket'] = DOCS_BUCKET
gcp['discovery_engine_location'] = DE_LOCATION
gcp['discovery_data_store_id'] = DATA_STORE_ID
gcp['discovery_engine_id'] = ENGINE_ID
gcp['discovery_serving_config'] = serving_config
config_path.write_text(json.dumps(config, indent=2))
print('✓ config.json updated:')
for k in ('docs_bucket', 'discovery_data_store_id', 'discovery_engine_id', 'discovery_serving_config'):
    print(f'  gcp.{k} = {gcp[k]}')

## Summary

| Resource | Details |
|---|---|
| **Search** | Vertex AI Search (Agent Search) via `google-cloud-discoveryengine` |
| **Data store** | `dataStores/orgiq-docs` (GENERIC, CONTENT_REQUIRED) |
| **Engine** | `engines/orgiq-search` (ENTERPRISE tier, LLM add-on) |
| **Docs source** | `gs://<project>-orgiq-documents/docs/*.md` |
| **Idle cost** | ~$0 — Discovery Engine bills per query + per GB indexed |
| **Comparison shape** | `normalize_de_results()` matches the AWS-side `retrieve()` dict |

**Next:** `05-model-armor.ipynb` — apply guardrails to generated answers.